# Part 2 — training strategy and augmentation

Use one experiment runner for all configurations. **A0 and A1** compare linear and cosine decay with the same single linear classifier. **A2** adds augmentation and label smoothing to A1 while preserving that classifier, so its comparison changes only the training inputs and targets. The optional MLP-head comparison remains separate under distinct run IDs.

All models start from scratch on the same Imagenette training subset. These custom 100-epoch runs use no teacher and do not reproduce the full published DeiT recipe. The historical Part 1 plateau-scheduler results are not the new A0 result. Execute each training cell manually when ready; each starts a full training run. The notebook has not been executed for full training.

In [ ]:
import json
import random
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset

from datasets.augmentation import build_image_transform, build_batch_transform
from datasets.imagenette import download_imagenette_dataset
from models.vision_transformer import build_vision_transformer, build_vision_transformer_single_classifier
from pipeline.evaluate import evaluate_model
from pipeline.train import train_model

SEED = 0
SPLIT_SEED = 0
EPOCHS = 100
WARMUP_EPOCHS = 5
BATCH_SIZE = 32
PEAK_LR = 5e-4
WEIGHT_DECAY = 0.01
LABEL_SMOOTHING = 0.1  # A2 only; existing clean controls keep unsmoothed targets.
DATA_ROOT = Path("data")
RESULTS_DIR = Path("artifacts") / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_UTC") / "part2_training"
MODEL_CONFIG = dict(image_size=224, patch_size=16, hidden_dim=192, num_heads=4,
                    num_layers=6, mlp_ratio=4, num_classes=10, dropout=0.1)
device = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")

def reset_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def save_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, default=lambda value: value.tolist()) + "\n")

print(f"Device: {device}; results: {RESULTS_DIR}")

## One fixed split and clean validation

Use only Imagenette's official **training** split, reserving a stratified 20% for internal validation. The official validation/test split is not loaded. A0, A1, and the MLP comparison use deterministic resize, center crop, and tensors in [0,1]. A2 uses a separate dataset instance with the same original training indices. Validation always uses the original clean instance.

Save the indices, image order, and class mapping once for every run in this session.

In [ ]:
pool = download_imagenette_dataset(root_dir=str(DATA_ROOT), split="train", image_size=224)
labels = np.array([label for _, label in pool._samples])
train_indices, validation_indices = train_test_split(
    np.arange(len(pool)), test_size=0.2, stratify=labels, random_state=SPLIT_SEED,
)
train_data = Subset(pool, train_indices.tolist())
validation_data = Subset(pool, validation_indices.tolist())
validation_loader = DataLoader(validation_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
np.savez(RESULTS_DIR / "split_indices.npz", train=train_indices, validation=validation_indices)
save_json(RESULTS_DIR / "dataset_manifest.json", {
    "dataset": "Imagenette", "distribution": "320px", "official_split": "train",
    "split_seed": SPLIT_SEED, "class_to_idx": pool.class_to_idx,
    "ordered_samples": [
        {"image": str(Path(path).relative_to(Path(pool._image_root))), "label": int(label)}
        for path, label in pool._samples
    ],
})
print(f"Training: {len(train_data)}; internal validation: {len(validation_data)}")

## One shared runner

Every call rebuilds the model and resets the initialization, dropout, sampling, and augmentation seeds. Each loader has a fresh generator with the same seed. This provides a matched starting point; stochastic augmentation can consume different random draws, and seeds do not guarantee bitwise reproducibility on every backend.

Keep 100 epochs, batch size 32, AdamW, actual peak LR 0.0005, weight decay 0.01, dropout 0.1, five warmup epochs, warmup LR `1e-6`, and minimum LR `1e-5`. There is no implicit batch-size scaling. Disable early stopping and retain the existing **highest validation accuracy** checkpoint rule in every run.

A0/A1 isolate the decay shape. A1/A2 isolate the augmentation-and-smoothing package. Image transforms run when the dataset loads an example; an optional batch transform runs once on the collated batch before the model. Validation keeps ordinary, unmixed ground-truth labels. Training loss with mixed, smoothed targets is not directly comparable to clean validation loss and need not approach zero.

In [ ]:
histories, summaries = {}, {}

def run_experiment(run_id, model_builder, *, scheduler, train_dataset, batch_transform=None):
    reset_seed(SEED)
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
        generator=torch.Generator().manual_seed(SEED), drop_last=False,
    )
    warmup_steps = WARMUP_EPOCHS * len(train_loader)
    model = model_builder(**MODEL_CONFIG)
    source_dataset = train_dataset.dataset if isinstance(train_dataset, Subset) else train_dataset
    run_dir = RESULTS_DIR / run_id
    checkpoint = run_dir / "best.pth"
    config = dict(
        run_id=run_id, model=MODEL_CONFIG, model_builder=model_builder.__name__,
        classifier_head=repr(model.mlp_head), student_seed=SEED, split_seed=SPLIT_SEED,
        epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=PEAK_LR,
        weight_decay=WEIGHT_DECAY, scheduler=scheduler, warmup_steps=warmup_steps,
        warmup_lr=1e-6, min_lr=1e-5, early_stopping_patience=None,
        supervision="one CLS head; cross-entropy against " +
                    ("class distributions from the batch transform" if batch_transform is not None else "ground-truth class indices"),
        evaluation="single-head prediction; clean labels", evaluation_diagnostics=True,
        checkpoint_rule="highest validation accuracy",
        image_transform=repr(source_dataset.transform), batch_transform=repr(batch_transform),
        device=str(device), torch_version=torch.__version__,
        split_indices="../split_indices.npz", dataset_manifest="../dataset_manifest.json",
    )
    save_json(run_dir / "config.json", config)
    history = train_model(
        model, train_loader, validation_loader, num_epochs=EPOCHS, learning_rate=PEAK_LR,
        weight_decay=WEIGHT_DECAY, device=device, checkpoint_path=checkpoint,
        scheduler=scheduler, warmup_steps=warmup_steps, warmup_lr=1e-6, min_lr=1e-5,
        early_stopping_patience=None, batch_transform=batch_transform,
    )
    metrics = evaluate_model(model, validation_loader, device, checkpoint_path=checkpoint, diagnostics=True)
    summary = dict(run_id=run_id, parameters=sum(p.numel() for p in model.parameters()),
                   epochs_run=len(history["train_loss"]), best_epoch=history["best_epoch"],
                   validation=metrics, training_seconds=sum(history["train_seconds"]),
                   checkpoint=str(checkpoint))
    save_json(run_dir / "history.json", history)
    save_json(run_dir / "summary.json", summary)
    histories[run_id], summaries[run_id] = history, summary
    model.cpu()
    print(f"{run_id}: validation accuracy={metrics['accuracy']:.4f}, "
          f"loss={metrics['loss']:.4f}, macro F1={metrics['macro_f1_score']:.4f}")
    return history, summary

## A0 — single linear classifier, linear decay

A0 establishes the controlled reference. It uses one CLS token and one linear classifier, with no augmentation or teacher.

In [ ]:
a0_history, a0_summary = run_experiment(
    "A0_linear", build_vision_transformer_single_classifier,
    scheduler="linear", train_dataset=train_data,
)

## A1 — single linear classifier, cosine decay

A1 uses the same model, initial seed, clean data, and warmup as A0. Only the post-warmup decay shape changes; it does not continue A0's checkpoint.

In [ ]:
a1_history, a1_summary = run_experiment(
    "A1_cosine", build_vision_transformer_single_classifier,
    scheduler="cosine_decay", train_dataset=train_data,
)

## A2 — add augmentation and label smoothing to A1

The image helper performs crop, horizontal flip, and torchvision RandAugment on the PIL image, then converts it to a tensor before random erasing. The batch helper chooses **MixUp or CutMix** for each training batch; these operations receive batched images and labels rather than individual PIL images.

Keep A1's **single linear classifier** here. Switching to the MLP builder at the same time would change both the head and augmentation. Use a separate augmented pool without replacing the clean pool or resplitting. After MixUp or CutMix, smooth the dataset targets once with epsilon 0.1: q = 0.9 * q + 0.1 / num_classes. Ordinary cross-entropy then uses those targets without additional smoothing. For ten classes, an unmixed one-hot target becomes 0.91 for its class and 0.01 for every other class. Validation and test targets remain original class indices, with unsmoothed NLL. No normalization is added, preserving the present input convention; torchvision's RandAugment and erasing remain adaptations, not the exact historical timm recipe.

Use a distinct run ID for this revised A2; previous unsmoothed A2 checkpoints do not acquire the effect of smoothing through evaluation. Their training must be repeated to measure this revised configuration.

In [ ]:
image_transform = build_image_transform(MODEL_CONFIG["image_size"])
batch_transform = build_batch_transform(MODEL_CONFIG["num_classes"], label_smoothing=LABEL_SMOOTHING)
augmented_pool = download_imagenette_dataset(
    root_dir=str(DATA_ROOT), split="train", image_size=224, transform=image_transform,
)
assert augmented_pool._samples == pool._samples, "Image order or class targets changed."
assert augmented_pool.class_to_idx == pool.class_to_idx, "Class mapping changed."
augmented_train_data = Subset(augmented_pool, train_indices.tolist())
a2_history, a2_summary = run_experiment(
    "A2_augmentation_smoothing", build_vision_transformer_single_classifier,
    scheduler="cosine_decay", train_dataset=augmented_train_data, batch_transform=batch_transform,
)

## Optional separate MLP-head comparison

These cells preserve your comparison with the two-layer GELU classification head. They keep clean training inputs and use unique IDs, so their checkpoints and histories cannot overwrite A0/A1. Compare MLP_A0 with MLP_A1 for scheduling within the MLP model, and each with the corresponding linear-head run for the effect of the head. They are not the reference for A2.

In [ ]:
mlp_a0_history, mlp_a0_summary = run_experiment(
    "MLP_A0_linear", build_vision_transformer,
    scheduler="linear", train_dataset=train_data,
)

In [ ]:
mlp_a1_history, mlp_a1_summary = run_experiment(
    "MLP_A1_cosine", build_vision_transformer,
    scheduler="cosine_decay", train_dataset=train_data,
)

## Compare completed runs

Plot only the runs executed in this kernel. Report clean validation results at each selected checkpoint, selected epochs, and actual training time. Single-seed differences remain exploratory; A2 changes a package of augmentations and label smoothing, so it cannot assign an individual contribution to each ingredient. Configurations record the actual head and both transform stages. No held-out test score is produced.

The same evaluator now also records 10-bin top-label ECE (confidence versus accuracy; lower is better), unscaled multiclass Brier score (squared probability error summed across classes; range 0–2, lower is better), and mean confidence and true-label NLL separately for correct and incorrect predictions. ECE is bin-dependent. Conditional confidence and NLL describe each model's own error subset; neither is a standalone model-ranking score. Empty groups are reported as `None`/JSON `null`. All confidence values and ECE use 0–1 units.


In [ ]:
if histories:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for run_id, history in histories.items():
        epochs = np.arange(1, len(history["train_loss"]) + 1)
        axes[0].plot(epochs, history["learning_rate"], label=run_id)
        axes[1].plot(epochs, history["val_loss"], label=run_id)
        axes[2].plot(epochs, history["val_accuracy"], label=run_id)
    for axis, title in zip(axes, ["LR at epoch start", "Clean validation loss", "Validation accuracy"]):
        axis.set(title=title, xlabel="Epoch")
        axis.grid(True)
        axis.legend()
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "training_comparison.png", dpi=160)
    plt.show()
    comparison_metrics = ("accuracy", "macro_f1_score", "loss", "brier_score", "ece",
                          "mean_confidence_incorrect")
    for run_id, summary in summaries.items():
        print(run_id, "selected epoch:", summary["best_epoch"],
              {name: summary["validation"][name] for name in comparison_metrics})
else:
    print("No runs have been executed in this kernel.")

## Evaluate existing checkpoints without training

Run the imports/setup cell, then this cell; skip all training cells. Fill `saved_runs` with the four original run directories and the matching model builders. Each directory must contain `best.pth`, and its parent must contain the saved `split_indices.npz` and `dataset_manifest.json`. This restores the checkpoint's model dimensions and original validation membership. The saved image order and class mapping are checked against the clean dataset.

This only evaluates the saved weights. It writes a separate `evaluation_diagnostics.json` beside each checkpoint, preserving the earlier summary and checkpoint. Summary numbers alone cannot supply these new probability metrics. The same additions are enabled automatically for future calls to `run_experiment`.


In [ ]:
saved_runs = {
    # "single_linear": (Path("artifacts/SESSION/part2_schedule/A0_linear"), build_vision_transformer_single_classifier),
    # "single_cosine": (Path("artifacts/SESSION/part2_schedule/A1_cosine"), build_vision_transformer_single_classifier),
    # "mlp_linear": (Path("artifacts/OTHER_SESSION/part2_schedule/A0_linear"), build_vision_transformer),
    # "mlp_cosine": (Path("artifacts/OTHER_SESSION/part2_schedule/A1_cosine"), build_vision_transformer),
}
checkpoint_metrics = {}
if saved_runs:
    evaluation_pool = download_imagenette_dataset(root_dir=str(DATA_ROOT), split="train", image_size=224)
    current_samples = [
        {"image": str(Path(path).relative_to(Path(evaluation_pool._image_root))), "label": int(label)}
        for path, label in evaluation_pool._samples
    ]
for name, (run_dir, model_builder) in saved_runs.items():
    run_dir = Path(run_dir)
    manifest = json.loads((run_dir.parent / "dataset_manifest.json").read_text())
    assert manifest["ordered_samples"] == current_samples, "Dataset image order or targets changed."
    assert manifest["class_to_idx"] == evaluation_pool.class_to_idx, "Class mapping changed."
    with np.load(run_dir.parent / "split_indices.npz") as split:
        saved_validation = Subset(evaluation_pool, split["validation"].tolist())
    saved_loader = DataLoader(saved_validation, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    checkpoint = run_dir / "best.pth"
    model_config = torch.load(checkpoint, map_location="cpu", weights_only=True)["model_config"]
    model = model_builder(**model_config)
    metrics = evaluate_model(model, saved_loader, device, checkpoint_path=checkpoint, diagnostics=True)
    checkpoint_metrics[name] = metrics
    save_json(run_dir / "evaluation_diagnostics.json", metrics)
    print(name, {key: metrics[key] for key in
                 ("accuracy", "macro_f1_score", "loss", "brier_score", "ece", "mean_confidence_incorrect")})
    model.cpu()
if not saved_runs:
    print("Add the saved run directories above to evaluate checkpoints without training.")
